In [32]:
"""
data_loader.py
--------------
Loads WiFi CSI data for the location-classification toy task.

Two modes:
  - real:      parse Widar3.0 / Widar1.0-2.0 style raw CSI .dat files
               (Intel 5300 format, read via the `csiread` package) and
               pull the location label straight out of the filename.
  - synthetic: generate CSI-like amplitude data with a controllable,
               location-dependent multipath signature, so the rest of
               the pipeline (features -> classifiers -> stress test)
               is runnable and debuggable before real data is on disk.

Widar filename convention (confirmed from the dataset's own bug-list
README): user-gesture-location-orientation-repetition-receiver.dat
  e.g. user2-6-4-4-2-r1.dat
       -> user=2, gesture=6, location=4, orientation=4, rep=2, receiver=1

If you end up using Widar1.0/2.0 or SignFi instead, their filenames
differ -- just edit FILENAME_RE and parse_label_from_filename below.
Everything downstream only cares that this function returns an int label.
"""

import re
import glob
import os
import numpy as np

N_ANTENNAS = 3
N_SUBCARRIERS = 30

FILENAME_RE = re.compile(
    r"user(?P<user>\d+)-(?P<gesture>\d+)-(?P<location>\d+)-"
    r"(?P<orientation>\d+)-(?P<rep>\d+)-r(?P<receiver>\d+)\.dat"
)


def parse_label_from_filename(path):
    """Extract the location field from a Widar-style filename.
    Returns None if the filename doesn't match, so callers can skip
    files that aren't in this naming convention (e.g. README/log files
    sitting in the same folder)."""
    m = FILENAME_RE.search(os.path.basename(path))
    if m is None:
        return None
    return int(m.group("location"))


def load_real_csi_file(path):
    """
    Parse one Intel-5300-format .dat file into an amplitude tensor
    of shape (n_packets, N_ANTENNAS, N_SUBCARRIERS).

    Requires: pip install csiread
    """
    import csiread
    csi = csiread.Intel(path, nrxnum=N_ANTENNAS, ntxnum=1, pl_size=10)
    csi.read()
    # csi.csi shape from csiread is (n_packets, n_subcarriers, n_rx, n_tx)
    amp = np.abs(csi.csi[:, :N_SUBCARRIERS, :N_ANTENNAS, 0])   # (T, 30, 3)
    amp = np.transpose(amp, (0, 2, 1))                          # (T, 3, 30)
    return amp


def load_real_dataset(data_dir, max_files=None):
    """Walk data_dir, parse every .dat file matching the naming
    convention, return (list_of_amplitude_arrays, list_of_int_labels).

    Files that fail to parse (corrupt / wrong format) are skipped with
    a printed warning rather than crashing the whole run -- Widar's own
    release notes admit a handful of empty/corrupt files exist.

    A MISSING csiread install is checked separately, up front, and
    raises immediately with install instructions -- otherwise every
    single file in the folder fails with the same ImportError and you
    get a wall of hundreds of identical [skip] lines instead of one
    useful message."""
    try:
        import csiread  # noqa: F401  (import-check only, used properly below)
    except ImportError as e:
        raise ImportError(
            "csiread is not installed in the Python environment your "
            "notebook kernel is actually running -- this is different from "
            "your system/base Python if you're on Windows with multiple "
            "Python or conda installs. Fix: run `%pip install csiread` "
            "(the %-magic, not `!pip install`, so it installs into the "
            "kernel's own environment rather than whatever `pip` happens "
            "to be first on PATH), then restart the kernel and rerun this "
            "cell. To double check which Python the kernel is using, run "
            "`import sys; print(sys.executable)` and compare it against "
            "the Python that `%pip install csiread` reports installing into."
        ) from e

    files = sorted(glob.glob(os.path.join(data_dir, "**", "*.dat"), recursive=True))
    if not files:
        raise FileNotFoundError(
            f"No .dat files found under {data_dir}. "
            f"Point DATA_DIR at your extracted Widar/SignFi folder."
        )

    X, y = [], []
    for f in files[:max_files]:
        label = parse_label_from_filename(f)
        if label is None:
            continue
        try:
            amp = load_real_csi_file(f)
        except Exception as e:
            print(f"[skip] {f}: {e}")
            continue
        if amp.shape[0] < 2:          # need at least 2 packets for delta features
            continue
        X.append(amp)
        y.append(label)

    print(f"[data] parsed {len(X)} usable files out of {len(files)} found")
    return X, y


def remap_labels(y):
    """
    Ensure labels are contiguous, zero-indexed integers.

    sklearn's classifiers (the k-NN baseline) don't care what the raw
    label values are -- they'll happily train and predict on {1,2,3,4,5}
    or {'kitchen','hallway',...} directly. PyTorch's CrossEntropyLoss is
    stricter: targets must be class indices in [0, n_classes). Widar's
    own `location` field runs 1-5, not 0-4, which is exactly what
    produced "IndexError: Target 5 is out of bounds" -- there were 5
    output neurons (indices 0-4) and a real label of 5 with nowhere to go.

    Call this once right after loading labels and use the remapped
    array everywhere downstream (k-NN, SNN, stress test) so indices
    stay consistent across the whole pipeline.

    Returns: (y_remapped: np.ndarray[int], label_map: {original_label: index})
    """
    unique_labels = sorted(set(y))
    label_map = {orig: idx for idx, orig in enumerate(unique_labels)}
    y_remapped = np.array([label_map[v] for v in y], dtype=int)
    return y_remapped, label_map


def generate_synthetic_dataset(n_locations=5, n_files_per_location=30,
                                n_packets=200, seed=0):
    """
    Fabricate CSI-like amplitude sequences so the pipeline is runnable
    end-to-end before real Widar/SignFi data is downloaded.

    Each location gets TWO location-specific ingredients:
      - a static "channel signature" (antenna x subcarrier gain pattern)
        -- mirrors DeepFi's Hypothesis 1: CSI is a stable snapshot at a
        fixed spot, different at different spots. This is what the
        no-model k-NN baseline (amplitude_snapshot) actually uses.
      - a slow, location-specific oscillation (its own frequency +
        phase per antenna/subcarrier) riding on top of the signature.
        This is what makes the *delta* features (features.delta_sequence)
        carry any location information at all.

    Why bother with the second part: an earlier version of this
    generator only had the static signature, and diffing removes a
    constant exactly -- so the delta/SNN pathway was training on pure
    noise (verified: mean delta norm was ~0.009 and statistically
    identical across every location). That's not a synthetic-data bug
    to shrug off, it's the same real limitation delta/event-based
    encoding has in practice (same criticism leveled at event cameras:
    they don't see absolute levels, only change). Real CSI has that
    "second ingredient" for free, because different locations really do
    have different multipath *dynamics*, not just different average
    gain. This generator fakes that in so the SNN pathway is actually
    testable -- swap in real data and this whole function goes away.
    """
    rng = np.random.default_rng(seed)
    signatures = rng.uniform(0.5, 2.0, size=(n_locations, N_ANTENNAS, N_SUBCARRIERS))
    dyn_freq = rng.uniform(0.01, 0.05, size=(n_locations, N_ANTENNAS, N_SUBCARRIERS))
    dyn_amp = 0.3

    X, y = [], []
    for loc in range(n_locations):
        for _ in range(n_files_per_location):
            t = np.arange(n_packets)
            phase = rng.uniform(0, 2 * np.pi, size=(N_ANTENNAS, N_SUBCARRIERS))
            dynamic = dyn_amp * np.sin(
                2 * np.pi * dyn_freq[loc][None, :, :] * t[:, None, None]
                + phase[None, :, :]
            )
            noise = rng.normal(0, 0.08, size=(n_packets, N_ANTENNAS, N_SUBCARRIERS))
            amp = signatures[loc][None, :, :] + dynamic + noise
            amp = np.clip(amp, 1e-3, None)
            X.append(amp)
            y.append(loc)
    return X, y

In [28]:
"""
features.py
-----------
Shared feature extraction for both the no-model baseline and the SNN.

Two feature views of the same raw CSI:

  - amplitude_snapshot: DeepFi-style, mean-normalized amplitude per
    (antenna, subcarrier), averaged over the packet window. A *static
    snapshot* view -- this is what fingerprinting systems (RADAR,
    Horus, DeepFi) actually store and match against.

  - delta_sequence: packet-to-packet amplitude change. An *event /
    motion* view -- a stationary target produces small deltas, a
    moving or vibrating one produces large, structured ones. This
    feeds the SNN and is also the thing the stress test in
    stress_test.py deliberately targets.
"""

import numpy as np


def amplitude_snapshot(amp_seq):
    """(T, n_antennas, n_subcarriers) -> flattened, unit-norm static
    fingerprint of shape (n_antennas * n_subcarriers,)."""
    mean_amp = amp_seq.mean(axis=0)                # (antennas, subcarriers)
    flat = mean_amp.flatten()
    norm = np.linalg.norm(flat) + 1e-8
    return flat / norm


def delta_sequence(amp_seq):
    """(T, n_antennas, n_subcarriers) -> (T-1, n_antennas*n_subcarriers)
    packet-to-packet amplitude deltas, flattened per antenna/subcarrier
    and scaled to unit std. This is the raw (pre-spike) input to the SNN."""
    diffs = np.diff(amp_seq, axis=0)                # (T-1, antennas, subcarriers)
    flat = diffs.reshape(diffs.shape[0], -1)        # (T-1, antennas*subcarriers)
    scale = np.std(flat) + 1e-8
    return flat / scale


def build_feature_matrix(X_raw, kind="amplitude"):
    """Convenience batch wrapper. kind in {'amplitude', 'delta'}.
    'amplitude' returns a stacked (N, D) array (fixed size per sample).
    'delta' returns a list of (T-1, D) arrays since T varies per file --
    pad/trim these yourself downstream (see snn_classifier.pad_or_trim)."""
    if kind == "amplitude":
        return np.stack([amplitude_snapshot(a) for a in X_raw])
    elif kind == "delta":
        return [delta_sequence(a) for a in X_raw]
    else:
        raise ValueError(f"unknown kind: {kind}")

In [29]:
"""
baseline_knn.py
---------------
The "no-model" baseline: classic fingerprint matching, in the spirit
of RADAR (Bahl & Padmanabhan) and Horus -- nearest-neighbor lookup
against stored amplitude snapshots, with zero learned parameters.
This is deliberately the simplest thing that could work, and is the
"simple classifier" the assignment asks for.
"""

import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


def train_and_eval_knn(X, y, k=1, test_size=0.3, seed=0, return_model=False):
    """
    X: (N, D) amplitude snapshot features (see features.amplitude_snapshot)
    y: (N,) int labels
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )
    clf = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print(f"[k-NN baseline] k={k}  test accuracy = {acc:.3f}")
    print("confusion matrix:\n", cm)

    if return_model:
        return clf, acc, cm, (X_train, X_test, y_train, y_test)
    return acc, cm

In [16]:
!pip install torch

In [5]:
!pip install snntorch

In [17]:
"""
snn_classifier.py
-----------------
A small spiking classifier over delta-encoded CSI, built the same way
you'd build the front end of DirectorSNN: encode a continuous signal
into spikes, run it through LIF layers, train with surrogate-gradient
backprop (snnTorch).

Encoding choice: delta-modulation, not rate coding. A packet-to-packet
amplitude change above a threshold fires a spike (sign encoded as two
channels: +delta and -delta) -- structurally the same delta-modulation
front end you've already built for analog-to-spike conversion on the
tapeout, just applied to CSI instead of an ADC waveform.

This choice is the actual thesis being tested here: a moving/vibrating
platform is defined by *continuous change*, so an encoding that fires
on change should, in principle, respond differently to motion-like
corruption than a scheme (the k-NN baseline) that stores a static
average. See stress_test.py for where that gets checked.

Requires: pip install snntorch torch
"""

import numpy as np
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from sklearn.model_selection import train_test_split


def delta_modulate(delta_seq, threshold=0.5):
    """
    delta_seq: (T-1, D) normalized amplitude deltas (see features.delta_sequence).
    Returns: (T-1, 2*D) spike train -- D "positive delta" channels and
    D "negative delta" channels, one spike per threshold crossing.
    """
    pos = (delta_seq > threshold).astype(np.float32)
    neg = (delta_seq < -threshold).astype(np.float32)
    return np.concatenate([pos, neg], axis=-1)


def pad_or_trim(spikes, target_len):
    """Force every sample to the same number of timesteps so they can
    be batched. Real recordings won't all have identical packet counts;
    this is the simplest fix -- revisit if you want windowing instead."""
    T = spikes.shape[0]
    if T >= target_len:
        return spikes[:target_len]
    pad = np.zeros((target_len - T, spikes.shape[1]), dtype=spikes.dtype)
    return np.concatenate([spikes, pad], axis=0)


class DeltaSNN(nn.Module):
    """
    Minimal LIF classifier: input_dim -> hidden -> n_classes.
    Same shape philosophy as the 26->512->5 DirectorSNN layer, just
    resized to this toy problem (2*antennas*subcarriers -> hidden -> n_locations).
    """
    def __init__(self, input_dim, hidden=256, n_classes=5, beta=0.9):
        super().__init__()
        spike_grad = surrogate.fast_sigmoid()
        self.fc1 = nn.Linear(input_dim, hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2 = nn.Linear(hidden, n_classes)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, spike_input):
        # spike_input: (T, batch, input_dim)
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem2_trace = []
    
        for t in range(spike_input.shape[0]):
            cur1 = self.fc1(spike_input[t])
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            mem2_trace.append(mem2)
        # NOTE: classification reads out the *membrane potential* trace of
        # the output layer, not spike counts. Spike-count readout looked
        # cleaner on paper but is fragile in practice: if the network ever
        # goes fully silent (no spikes anywhere), the surrogate gradient
        # has nothing left to push on and training gets stuck in a dead,
        # zero-output state -- which is exactly what happened during
        # testing here (loss flatlined at ln(n_classes), every output
        # frozen at 0). The membrane potential is a continuous leaky
        # integration of the input current every timestep, so it keeps
        # a live gradient signal even while spk1/spk2 are sparse or zero.
        return torch.stack(mem2_trace, dim=0).mean(dim=0)  # (batch, n_classes)


def train_snn(X_spikes, y, n_classes, hidden=256, epochs=30, lr=1e-3,
              seed=0, test_size=0.3, verbose=True):
    """
    X_spikes: list/array of (T, 2*D) spike arrays, already padded to equal T
               (see pad_or_trim).
    y: list/array of int labels.
    Returns: (trained_model, final_test_accuracy, (X_tr, X_te, y_tr, y_te))
    """
    X_arr = np.stack(X_spikes)                      # (N, T, 2D)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_arr, y, test_size=test_size, random_state=seed, stratify=y
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = DeltaSNN(input_dim=X_arr.shape[-1], hidden=hidden, n_classes=n_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    def to_tensor(X, y_):
        Xt = torch.tensor(X, dtype=torch.float32).permute(1, 0, 2).to(device)  # (T, N, 2D)
        yt = torch.tensor(y_, dtype=torch.long).to(device)
        return Xt, yt

    Xtr_t, ytr_t = to_tensor(X_tr, y_tr)
    Xte_t, yte_t = to_tensor(X_te, y_te)

    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(Xtr_t)
        loss = loss_fn(out, ytr_t)
        loss.backward()
        opt.step()
        if verbose and (ep % 5 == 0 or ep == epochs - 1):
            model.eval()
            with torch.no_grad():
                acc = (model(Xte_t).argmax(1) == yte_t).float().mean().item()
            print(f"[SNN] epoch {ep:02d}  loss={loss.item():.3f}  test_acc={acc:.3f}")

    model.eval()
    with torch.no_grad():
        final_acc = (model(Xte_t).argmax(1) == yte_t).float().mean().item()
    return model, final_acc, (X_tr, X_te, y_tr, y_te)


def snn_predict(model, X_spikes):
    """Run inference on a fresh list of (T, 2D) spike arrays (already
    padded to the same T the model was trained with)."""
    device = next(model.parameters()).device
    X_arr = np.stack(X_spikes)
    Xt = torch.tensor(X_arr, dtype=torch.float32).permute(1, 0, 2).to(device)
    model.eval()
    with torch.no_grad():
        preds = model(Xt).argmax(1).cpu().numpy()
    return preds

In [18]:
"""
stress_test.py
--------------
Synthetic motion-corruption stress test.

There's no real drone CSI dataset to grab for this assignment, so
instead: take the *same* static CSI test data and corrupt it the way a
moving/tilting/vibrating platform would, then measure how much each
classifier degrades. This directly answers "what actually changes
when the thing collecting CSI is a drone" with numbers instead of
hand-waving.

Three corruptions, each targeting a specific assumption a
static-target CSI system makes:

1. orientation_shift   -> circularly rolls the antenna axis, mimicking
   the antenna array rotating relative to the incident angle of
   arrival. Breaks: any AoA/steering-vector model (e.g. SpotFi) that
   assumes fixed, known antenna geometry relative to incoming multipath.

2. vibration_jitter     -> adds a high-frequency sinusoidal ripple to
   amplitude across packets, mimicking propeller-induced vibration.
   Breaks: the assumption (DeepFi's own Hypothesis 1) that CSI is
   stable over a short packet burst at a fixed location.

3. doppler_phase_ramp   -> adds a linearly growing amplitude/phase
   offset across packets, mimicking a continuously changing path
   length as the platform moves during the observation window.
   Breaks: SpotFi-style STO sanitization, which fits and removes a
   *constant* linear-phase term, assuming the platform doesn't move
   during the burst.

Modify or add corruptions freely -- the point is the comparison
across conditions, not these three specifically.
"""

import numpy as np

RNG_SEED = 0


def orientation_shift(amp_seq, shift=1):
    """Roll along the antenna axis (axis=1)."""
    return np.roll(amp_seq, shift=shift, axis=1)


def vibration_jitter(amp_seq, freq_hz=200, packet_rate_hz=1000,
                      amplitude=0.15, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    T = amp_seq.shape[0]
    t = np.arange(T) / packet_rate_hz
    ripple = amplitude * np.sin(2 * np.pi * freq_hz * t)
    ripple = ripple + rng.normal(0, 0.02, size=T)
    return amp_seq * (1 + ripple[:, None, None])


def doppler_phase_ramp(amp_seq, ramp_strength=0.02):
    T = amp_seq.shape[0]
    ramp = np.linspace(0, ramp_strength * T, T)
    return amp_seq * (1 + ramp[:, None, None])


CORRUPTIONS = {
    "orientation_shift": orientation_shift,
    "vibration_jitter": vibration_jitter,
    "doppler_phase_ramp": doppler_phase_ramp,
}


def apply_corruption(X_raw, name, **kwargs):
    fn = CORRUPTIONS[name]
    return [fn(a, **kwargs) for a in X_raw]


def run_stress_test(X_raw_test, y_test, eval_fn_baseline, eval_fn_snn):
    """
    eval_fn_baseline(X_raw_list, y_list) -> accuracy   (feature-extract + knn.predict)
    eval_fn_snn(X_raw_list, y_list)      -> accuracy   (delta-encode + snn forward)

    Returns dict: condition_name -> (baseline_acc, snn_acc)
    """
    results = {}
    results["clean"] = (eval_fn_baseline(X_raw_test, y_test),
                         eval_fn_snn(X_raw_test, y_test))

    for name in CORRUPTIONS:
        X_corrupt = apply_corruption(X_raw_test, name)
        results[name] = (eval_fn_baseline(X_corrupt, y_test),
                          eval_fn_snn(X_corrupt, y_test))

    print(f"\n{'condition':<20}{'no-model (kNN)':<18}{'SNN':<10}")
    for k, (b, s) in results.items():
        print(f"{k:<20}{b:<18.3f}{s:<10.3f}")
    return results

In [6]:
"""
main.py
-------
Orchestrates the full Part 2 pipeline:

  1. load data (synthetic by default -- flip USE_SYNTHETIC once you
     have a real Widar/SignFi download on disk)
  2. train the no-model k-NN fingerprint baseline
  3. train the delta-encoded SNN classifier
  4. run the synthetic motion-corruption stress test on both

--------------------------------------------------------------------
Works in two setups:
  (a) as real separate .py files (e.g. your /code submission folder)
      -> the imports below succeed normally.
  (b) pasted into notebook cells, one file per cell, run in order:
      data_loader -> features -> baseline_knn -> snn_classifier
      -> stress_test -> main (this cell) -> a cell calling main()
      -> there's no data_loader.py etc. on disk for Python to find,
      so the imports below fail, and the except clause below just
      assumes those names already exist in the notebook's global
      namespace from the earlier cells having been run.

If you rerun a module cell after editing it, rerun this main cell too
before calling main() again -- case (b) doesn't re-import anything
for you, so it won't otherwise pick up the change.
--------------------------------------------------------------------

Run (script version): python main.py
Run (notebook version): run this cell, then call main() in the next cell
"""

import numpy as np
from sklearn.model_selection import train_test_split

try:
    from data_loader import generate_synthetic_dataset, load_real_dataset, remap_labels
    from features import amplitude_snapshot, delta_sequence
    from baseline_knn import train_and_eval_knn
    from snn_classifier import delta_modulate, pad_or_trim, train_snn, snn_predict
    from stress_test import run_stress_test
except ImportError:
    # Setup (b): these files don't exist on disk as separate modules --
    # assume every name above is already defined in this notebook's
    # global namespace because its cell already ran. Nothing to do.
    pass

# ------------------------------------------------------------------
# CONFIG -- edit these first
# ------------------------------------------------------------------
USE_SYNTHETIC = False             # False once real data is downloaded
DATA_DIR = r"D:\RealDownloads\20181117\20181117\user4"      # only used if USE_SYNTHETIC = False
N_LOCATIONS = 5                  # only used for synthetic generation
SPIKE_SEQ_LEN = 60               # pad/trim delta-spike sequences to this many steps
DELTA_THRESHOLD = 0.5
TEST_SIZE = 0.3
SEED = 0


def main():
    # 1. Load data ----------------------------------------------------
    if USE_SYNTHETIC:
        print("[data] using synthetic CSI (edit USE_SYNTHETIC=False once real data is ready)")
        X_raw, y = generate_synthetic_dataset(n_locations=N_LOCATIONS, seed=SEED)
    else:
        X_raw, y = load_real_dataset(DATA_DIR)

    y = np.array(y)
    y, label_map = remap_labels(y)
    n_classes = len(label_map)
    print(f"[data] {len(X_raw)} samples, {n_classes} location classes")
    print(f"[data] label remap (original -> model index): {label_map}\n")

    # 2. No-model baseline ---------------------------------------------
    print("=== No-model baseline (k-NN fingerprint match) ===")
    X_amp = np.stack([amplitude_snapshot(a) for a in X_raw])
    knn_model, knn_acc, knn_cm, _ = train_and_eval_knn(
        X_amp, y, k=1, test_size=TEST_SIZE, seed=SEED, return_model=True
    )

    # 3. SNN on delta-encoded CSI ---------------------------------------
    print("\n=== SNN (delta-encoded LIF classifier) ===")
    X_spikes = []
    for a in X_raw:
        d = delta_sequence(a)
        s = delta_modulate(d, threshold=DELTA_THRESHOLD)
        s = pad_or_trim(s, SPIKE_SEQ_LEN)
        X_spikes.append(s)

    snn_model, snn_acc, _ = train_snn(
        X_spikes, y, n_classes=n_classes, epochs=80, seed=SEED, test_size=TEST_SIZE
    )

    # 4. Stress test: simulate a moving/vibrating collector --------------
    print("\n=== Stress test: static-trained classifiers vs. simulated motion ===")

    X_raw_tr, X_raw_te, y_tr, y_te = train_test_split(
        X_raw, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )

    def eval_baseline(X_raw_list, y_list):
        feats = np.stack([amplitude_snapshot(a) for a in X_raw_list])
        preds = knn_model.predict(feats)
        return (preds == np.array(y_list)).mean()

    def eval_snn(X_raw_list, y_list):
        spikes = []
        for a in X_raw_list:
            d = delta_sequence(a)
            s = delta_modulate(d, threshold=DELTA_THRESHOLD)
            s = pad_or_trim(s, SPIKE_SEQ_LEN)
            spikes.append(s)
        preds = snn_predict(snn_model, spikes)
        return (preds == np.array(y_list)).mean()

    run_stress_test(X_raw_te, y_te, eval_baseline, eval_snn)

    print("\n[done] baseline test acc = {:.3f}, SNN test acc = {:.3f}".format(knn_acc, snn_acc))


# --------------------------------------------------------------------
# Script version: `python main.py` runs this automatically (__name__
# is "__main__" for a script). Notebook version: __name__ is also
# "__main__" for a normally-run cell, so this fires too -- but if your
# notebook setup executes cells any other way (papermill, %run with
# args, etc.) and this silently doesn't fire, just call main() directly
# in the next cell instead.
# --------------------------------------------------------------------
if __name__ == "__main__":
    main()

[data] parsed 4500 usable files out of 4500 found
[data] 4500 samples, 5 location classes
[data] label remap (original -> model index): {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2, np.int64(4): 3, np.int64(5): 4}

=== No-model baseline (k-NN fingerprint match) ===
[k-NN baseline] k=1  test accuracy = 0.935
confusion matrix:
 [[253   0   6   5   6]
 [  1 258   2   1   8]
 [  4   4 251   6   5]
 [  1   2   1 254  12]
 [  1   3   4  16 246]]

=== SNN (delta-encoded LIF classifier) ===
[SNN] epoch 00  loss=1.660  test_acc=0.213
[SNN] epoch 05  loss=1.606  test_acc=0.235
[SNN] epoch 10  loss=1.605  test_acc=0.235
[SNN] epoch 15  loss=1.597  test_acc=0.230
[SNN] epoch 20  loss=1.590  test_acc=0.250
[SNN] epoch 25  loss=1.583  test_acc=0.259
[SNN] epoch 30  loss=1.577  test_acc=0.266
[SNN] epoch 35  loss=1.571  test_acc=0.265
[SNN] epoch 40  loss=1.567  test_acc=0.250
[SNN] epoch 45  loss=1.560  test_acc=0.273
[SNN] epoch 50  loss=1.559  test_acc=0.268
[SNN] epoch 55  loss=1.556  test_acc

In [13]:
!pip install csiread


In [19]:
import numpy as np
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from sklearn.model_selection import train_test_split


def pad_or_trim(deltas, target_len):
    """Force every sequence sample to target_len timesteps."""
    T = deltas.shape[0]
    if T >= target_len:
        return deltas[:target_len]
    pad = np.zeros((target_len - T, deltas.shape[1]), dtype=deltas.dtype)
    return np.concatenate([deltas, pad], axis=0)


class TrainableAdaptiveDeltaEncoder(nn.Module):
    """
    In-graph delta-modulation encoder with learnable per-channel base thresholds 
    and dynamic running-intensity adaptation.
    """
    def __init__(self, num_channels, init_threshold=0.5, adapt_decay=0.9, adapt_scale=0.1):
        super().__init__()
        # Learnable channel-wise threshold parameters (initialized in softplus space)
        init_val = np.log(np.exp(init_threshold) - 1.0)
        self.raw_thresh_pos = nn.Parameter(torch.full((num_channels,), init_val, dtype=torch.float32))
        self.raw_thresh_neg = nn.Parameter(torch.full((num_channels,), init_val, dtype=torch.float32))
        
        self.adapt_decay = adapt_decay
        self.adapt_scale = adapt_scale
        self.spike_grad = surrogate.fast_sigmoid()

    def forward(self, delta_seq):
        # delta_seq: (T, batch, D)
        T, batch_size, D = delta_seq.shape
        device = delta_seq.device
        
        # Ensure positive thresholds via softplus
        thresh_pos_base = nn.functional.softplus(self.raw_thresh_pos)
        thresh_neg_base = nn.functional.softplus(self.raw_thresh_neg)
        
        adapt_state = torch.zeros(batch_size, D, device=device)
        spikes_pos = []
        spikes_neg = []

        for t in range(T):
            dx = delta_seq[t]  # (batch, D)
            
            # Dynamic thresholding: base threshold + state-dependent adaptation
            th_pos = thresh_pos_base + self.adapt_scale * adapt_state
            th_neg = thresh_neg_base + self.adapt_scale * adapt_state
            
            # Spike generation using surrogate gradient
            spk_p = self.spike_grad(dx - th_pos)
            spk_n = self.spike_grad(-dx - th_neg)
            
            spikes_pos.append(spk_p)
            spikes_neg.append(spk_n)
            
            # Update adaptation state using running average of absolute delta intensity
            adapt_state = self.adapt_decay * adapt_state + (1.0 - self.adapt_decay) * torch.abs(dx)

        pos_tensor = torch.stack(spikes_pos, dim=0)
        neg_tensor = torch.stack(spikes_neg, dim=0)
        
        return torch.cat([pos_tensor, neg_tensor], dim=-1)  # (T, batch, 2*D)


class AdaptiveLIF(nn.Module):
    """
    Leaky Integrate-and-Fire neuron with learnable base threshold 
    and spike-frequency threshold adaptation.
    """
    def __init__(self, num_neurons, beta=0.9, init_v_th=1.0, adapt_decay=0.9, adapt_scale=0.2):
        super().__init__()
        self.beta = beta
        self.adapt_decay = adapt_decay
        self.adapt_scale = adapt_scale
        self.spike_grad = surrogate.fast_sigmoid()
        
        # Learnable base firing threshold per neuron
        init_val = np.log(np.exp(init_v_th) - 1.0)
        self.raw_v_th = nn.Parameter(torch.full((num_neurons,), init_val, dtype=torch.float32))

    def init_states(self, batch_size, device):
        num_neurons = self.raw_v_th.shape[0]
        mem = torch.zeros(batch_size, num_neurons, device=device)
        adapt_state = torch.zeros(batch_size, num_neurons, device=device)
        return mem, adapt_state

    def forward(self, input_current, mem, adapt_state):
        base_v_th = nn.functional.softplus(self.raw_v_th)
        v_th = base_v_th + self.adapt_scale * adapt_state
        
        # LIF integration and firing
        mem = self.beta * mem + input_current
        spk = self.spike_grad(mem - v_th)
        mem = mem - spk * v_th  # Soft reset proportional to threshold
        
        # Dynamic threshold adaptation (increases with firing activity)
        adapt_state = self.adapt_decay * adapt_state + spk
        
        return spk, mem, adapt_state

class DeltaSNN(nn.Module):
    """
    Spiking classifier with integrated trainable delta encoder 
    and adaptive LIF layers, using Spike Count (Winner-Takes-All) readout.
    """
    def __init__(self, raw_input_dim, hidden=256, n_classes=5, beta=0.9, init_encoder_thresh=0.5):
        super().__init__()
        self.encoder = TrainableAdaptiveDeltaEncoder(
            num_channels=raw_input_dim, 
            init_threshold=init_encoder_thresh
        )
        
        encoded_dim = 2 * raw_input_dim
        self.fc1 = nn.Linear(encoded_dim, hidden)
        self.lif1 = AdaptiveLIF(num_neurons=hidden, beta=beta)
        
        self.fc2 = nn.Linear(hidden, n_classes)
        # Re-introduce the spiking LIF neuron for the output layer
        self.lif2 = AdaptiveLIF(num_neurons=n_classes, beta=beta)

    def forward(self, raw_delta_seq):
        # raw_delta_seq shape: (T, batch, raw_input_dim)
        spike_input = self.encoder(raw_delta_seq)  # (T, batch, 2 * raw_input_dim)
        
        batch_size = raw_delta_seq.shape[1]
        device = raw_delta_seq.device
        
        mem1, adapt1 = self.lif1.init_states(batch_size, device)
        mem2, adapt2 = self.lif2.init_states(batch_size, device)
        
        spk2_trace = []

        for t in range(spike_input.shape[0]):
            cur1 = self.fc1(spike_input[t])
            spk1, mem1, adapt1 = self.lif1(cur1, mem1, adapt1)
            
            cur2 = self.fc2(spk1)
            # Output layer now generates spikes
            spk2, mem2, adapt2 = self.lif2(cur2, mem2, adapt2)
            
            spk2_trace.append(spk2)

        # Sum over time (T) to get total spike count per class -> (batch, n_classes)
        return torch.stack(spk2_trace, dim=0).sum(dim=0)

def train_snn2(X_deltas, y, n_classes, hidden=256, epochs=30, lr=1e-3,
              seed=0, test_size=0.3, verbose=True):
    """
    X_deltas: list or numpy array of (T, D) continuous delta sequences.
    y: list or numpy array of integer labels.
    """
    X_arr = np.stack(X_deltas)  # (N, T, D)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_arr, y, test_size=test_size, random_state=seed, stratify=y
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    raw_input_dim = X_arr.shape[-1]
    
    model = DeltaSNN(raw_input_dim=raw_input_dim, hidden=hidden, n_classes=n_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    def to_tensor(X, y_):
        Xt = torch.tensor(X, dtype=torch.float32).permute(1, 0, 2).to(device)  # (T, N, D)
        yt = torch.tensor(y_, dtype=torch.long).to(device)
        return Xt, yt

    Xtr_t, ytr_t = to_tensor(X_tr, y_tr)
    Xte_t, yte_t = to_tensor(X_te, y_te)

    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(Xtr_t)
        loss = loss_fn(out, ytr_t)
        loss.backward()
        opt.step()

        if verbose and (ep % 5 == 0 or ep == epochs - 1):
            model.eval()
            with torch.no_grad():
                acc = (model(Xte_t).argmax(1) == yte_t).float().mean().item()
            print(f"[SNN] epoch {ep:02d}  loss={loss.item():.3f}  test_acc={acc:.3f}")

    model.eval()
    with torch.no_grad():
        final_acc = (model(Xte_t).argmax(1) == yte_t).float().mean().item()
        
    return model, final_acc, (X_tr, X_te, y_tr, y_te)


def snn_predict2(model, X_deltas):
    """Run inference on fresh (T, D) continuous delta sequences."""
    device = next(model.parameters()).device
    X_arr = np.stack(X_deltas)
    Xt = torch.tensor(X_arr, dtype=torch.float32).permute(1, 0, 2).to(device)
    model.eval()
    with torch.no_grad():
        preds = model(Xt).argmax(1).cpu().numpy()
    return preds

In [21]:
"""
main.py
-------
Orchestrates the full Part 2 pipeline:

  1. load data (synthetic by default -- flip USE_SYNTHETIC once you
     have a real Widar/SignFi download on disk)
  2. train the no-model k-NN fingerprint baseline
  3. train the delta-encoded SNN classifier (with in-graph trainable delta thresholds)
  4. run the synthetic motion-corruption stress test on both

--------------------------------------------------------------------
Works in two setups:
  (a) as real separate .py files (e.g. your /code submission folder)
      -> the imports below succeed normally.
  (b) pasted into notebook cells, one file per cell, run in order:
      data_loader -> features -> baseline_knn -> snn_classifier
      -> stress_test -> main (this cell) -> a cell calling main()
      -> there's no data_loader.py etc. on disk for Python to find,
      so the imports below fail, and the except clause below just
      assumes those names already exist in the notebook's global
      namespace from the earlier cells having been run.

If you rerun a module cell after editing it, rerun this main cell too
before calling main() again -- case (b) doesn't re-import anything
for you, so it won't otherwise pick up the change.
--------------------------------------------------------------------

Run (script version): python main.py
Run (notebook version): run this cell, then call main() in the next cell
"""

import numpy as np
from sklearn.model_selection import train_test_split

try:
    from data_loader import generate_synthetic_dataset, load_real_dataset, remap_labels
    from features import amplitude_snapshot, delta_sequence
    from baseline_knn import train_and_eval_knn
    from snn_classifier import pad_or_trim, train_snn, snn_predict
    from stress_test import run_stress_test
except ImportError:
    # Setup (b): these files don't exist on disk as separate modules --
    # assume every name above is already defined in this notebook's
    # global namespace because its cell already ran. Nothing to do.
    pass

# ------------------------------------------------------------------
# CONFIG -- edit these first
# ------------------------------------------------------------------
USE_SYNTHETIC = False             # False once real data is downloaded
DATA_DIR = r"D:\RealDownloads\20181117\20181117\user4"      # only used if USE_SYNTHETIC = False
N_LOCATIONS = 5                  # only used for synthetic generation
SPIKE_SEQ_LEN = 60               # pad/trim delta sequences to this many steps
DELTA_THRESHOLD = 0.3            # passed as initial base threshold to trainable encoder
TEST_SIZE = 0.3
SEED = 0


def main():
    # 1. Load data ----------------------------------------------------
    if USE_SYNTHETIC:
        print("[data] using synthetic CSI (edit USE_SYNTHETIC=False once real data is ready)")
        X_raw, y = generate_synthetic_dataset(n_locations=N_LOCATIONS, seed=SEED)
    else:
        X_raw, y = load_real_dataset(DATA_DIR)

    y = np.array(y)
    y, label_map = remap_labels(y)
    n_classes = len(label_map)
    print(f"[data] {len(X_raw)} samples, {n_classes} location classes")
    print(f"[data] label remap (original -> model index): {label_map}\n")

    # 2. No-model baseline ---------------------------------------------
    print("=== No-model baseline (k-NN fingerprint match) ===")
    X_amp = np.stack([amplitude_snapshot(a) for a in X_raw])
    knn_model, knn_acc, knn_cm, _ = train_and_eval_knn(
        X_amp, y, k=1, test_size=TEST_SIZE, seed=SEED, return_model=True
    )

    # 3. SNN on delta-encoded CSI ---------------------------------------
    print("\n=== SNN (delta-encoded LIF classifier) ===")
    X_deltas = []
    for a in X_raw:
        d = delta_sequence(a)
        d = pad_or_trim(d, SPIKE_SEQ_LEN)
        X_deltas.append(d)

    snn_model2, snn_acc, _ = train_snn2(
        X_deltas, y, n_classes=n_classes, epochs=80, seed=SEED, test_size=TEST_SIZE
    )

    # 4. Stress test: simulate a moving/vibrating collector --------------
    print("\n=== Stress test: static-trained classifiers vs. simulated motion ===")

    X_raw_tr, X_raw_te, y_tr, y_te = train_test_split(
        X_raw, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )

    def eval_baseline(X_raw_list, y_list):
        feats = np.stack([amplitude_snapshot(a) for a in X_raw_list])
        preds = knn_model.predict(feats)
        return (preds == np.array(y_list)).mean()

    def eval_snn(X_raw_list, y_list):
        deltas = []
        for a in X_raw_list:
            d = delta_sequence(a)
            d = pad_or_trim(d, SPIKE_SEQ_LEN)
            deltas.append(d)
        preds = snn_predict2(snn_model2, deltas)
        return (preds == np.array(y_list)).mean()

    run_stress_test(X_raw_te, y_te, eval_baseline, eval_snn)

    print("\n[done] baseline test acc = {:.3f}, SNN test acc = {:.3f}".format(knn_acc, snn_acc))


if __name__ == "__main__":
    main()

[data] parsed 4500 usable files out of 4500 found
[data] 4500 samples, 5 location classes
[data] label remap (original -> model index): {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2, np.int64(4): 3, np.int64(5): 4}

=== No-model baseline (k-NN fingerprint match) ===
[k-NN baseline] k=1  test accuracy = 0.935
confusion matrix:
 [[253   0   6   5   6]
 [  1 258   2   1   8]
 [  4   4 251   6   5]
 [  1   2   1 254  12]
 [  1   3   4  16 246]]

=== SNN (delta-encoded LIF classifier) ===
[SNN] epoch 00  loss=1.609  test_acc=0.212
[SNN] epoch 05  loss=1.609  test_acc=0.200
[SNN] epoch 10  loss=1.610  test_acc=0.196
[SNN] epoch 15  loss=1.609  test_acc=0.200
[SNN] epoch 20  loss=1.611  test_acc=0.201
[SNN] epoch 25  loss=1.608  test_acc=0.200
[SNN] epoch 30  loss=1.603  test_acc=0.200
[SNN] epoch 35  loss=1.605  test_acc=0.200
[SNN] epoch 40  loss=1.601  test_acc=0.200
[SNN] epoch 45  loss=1.600  test_acc=0.218
[SNN] epoch 50  loss=1.603  test_acc=0.200
[SNN] epoch 55  loss=1.601  test_acc

In [30]:
"""
snn_classifier.py
-----------------
A small spiking classifier over delta-encoded CSI, built the same way
you'd build the front end of DirectorSNN: encode a continuous signal
into spikes, run it through LIF layers, train with surrogate-gradient
backprop (snnTorch).

Encoding choice: delta-modulation, not rate coding. A packet-to-packet
amplitude change above a threshold fires a spike (sign encoded as two
channels: +delta and -delta) -- structurally the same delta-modulation
front end you've already built for analog-to-spike conversion on the
tapeout, just applied to CSI instead of an ADC waveform.

This choice is the actual thesis being tested here: a moving/vibrating
platform is defined by *continuous change*, so an encoding that fires
on change should, in principle, respond differently to motion-like
corruption than a scheme (the k-NN baseline) that stores a static
average. See stress_test.py for where that gets checked.

Requires: pip install snntorch torch
"""

import numpy as np
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from sklearn.model_selection import train_test_split


def delta_modulate(delta_seq, threshold=0.5):
    """
    delta_seq: (T-1, D) normalized amplitude deltas (see features.delta_sequence).
    Returns: (T-1, 2*D) spike train -- D "positive delta" channels and
    D "negative delta" channels, one spike per threshold crossing.

    This uses ONE global threshold for every channel. That was fine on
    the synthetic data (every channel built to the same scale by
    construction), but real CSI channels don't share a scale -- some
    subcarrier/antenna combinations sit near a fade and swing hugely,
    others barely move. A single global cutoff either starves the quiet
    channels of spikes entirely or saturates the noisy ones into firing
    on every packet, and in both cases destroys the information the
    threshold was supposed to preserve. See adaptive_delta_modulate
    below for the per-channel fix.
    """
    pos = (delta_seq > threshold).astype(np.float32)
    neg = (delta_seq < -threshold).astype(np.float32)
    return np.concatenate([pos, neg], axis=-1)


def compute_adaptive_thresholds(train_delta_seqs, k=1.0, min_threshold=1e-3):
    """
    Per-channel threshold = k * (that channel's delta std, measured
    ONLY on the training set). Returns a (D,) array instead of a single
    scalar, so each of the 90 antenna/subcarrier channels gets a cutoff
    matched to its own dynamic range instead of one global guess.

    Computed from training data only (never touch the test set here) --
    fitting a threshold from test-set statistics would be a real
    leakage bug, exactly the kind of thing that quietly inflates
    reported numbers.

    train_delta_seqs: list of (T-1, D) raw delta arrays from
    features.delta_sequence, computed on the TRAIN split only.
    """
    all_deltas = np.concatenate(train_delta_seqs, axis=0)         # (sum_T, D)
    per_channel_std = all_deltas.std(axis=0)
    thresholds = np.maximum(k * per_channel_std, min_threshold)
    return thresholds.astype(np.float32)


def adaptive_delta_modulate(delta_seq, thresholds):
    """Same idea as delta_modulate, but thresholds is a (D,) per-channel
    vector (from compute_adaptive_thresholds) instead of one scalar."""
    pos = (delta_seq > thresholds[None, :]).astype(np.float32)
    neg = (delta_seq < -thresholds[None, :]).astype(np.float32)
    return np.concatenate([pos, neg], axis=-1)


def graded_delta_encode(delta_seq, clip=3.0):
    """
    Non-binary alternative: instead of collapsing each channel's delta
    to a single above/below-threshold bit, feed the clipped magnitude
    directly as a graded input current (still split into +/- channels
    so the network sees direction, same 2*D output shape as the spike
    encodings above -- DeltaSNN doesn't need to know which encoding
    produced its input).

    Worth comparing directly against both delta_modulate variants: if
    graded does meaningfully better than adaptive binary, the bottleneck
    was information lost at the binarization step, not the threshold
    calibration. If it does about the same, binarization wasn't the
    problem.
    delta_seq: (T-1, D) -> (T-1, 2*D), values in [0, clip].
    """
    clipped = np.clip(delta_seq, -clip, clip)
    pos = np.clip(clipped, 0, None)
    neg = np.clip(-clipped, 0, None)
    return np.concatenate([pos, neg], axis=-1).astype(np.float32)


def pad_or_trim(spikes, target_len):
    """Force every sample to the same number of timesteps so they can
    be batched. Real recordings won't all have identical packet counts;
    this is the simplest fix -- revisit if you want windowing instead."""
    T = spikes.shape[0]
    if T >= target_len:
        return spikes[:target_len]
    pad = np.zeros((target_len - T, spikes.shape[1]), dtype=spikes.dtype)
    return np.concatenate([spikes, pad], axis=0)


class DeltaSNN(nn.Module):
    """
    LIF classifier: input_dim -> hidden [-> hidden2] -> n_classes.
    Same shape philosophy as the 26->512->5 DirectorSNN layer, resized
    to this problem. hidden2=None (default) keeps the original single
    hidden layer; set it to add a second LIF layer for more capacity
    on real data, which is noisier and higher-dimensional in practice
    than the synthetic construction this started on.
    """
    def __init__(self, input_dim, hidden=256, hidden2=None, n_classes=5,
                 beta=0.9, dropout=0.0):
        super().__init__()
        spike_grad = surrogate.fast_sigmoid()
        self.hidden2 = hidden2
        self.fc1 = nn.Linear(input_dim, hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.drop1 = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        if hidden2:
            self.fc_mid = nn.Linear(hidden, hidden2)
            self.lif_mid = snn.Leaky(beta=beta, spike_grad=spike_grad)
            self.fc2 = nn.Linear(hidden2, n_classes)
        else:
            self.fc2 = nn.Linear(hidden, n_classes)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, spike_input):
        # spike_input: (T, batch, input_dim)
        mem1 = self.lif1.init_leaky()
        mem_mid = self.lif_mid.init_leaky() if self.hidden2 else None
        mem2 = self.lif2.init_leaky()
        mem2_trace = []
        for t in range(spike_input.shape[0]):
            cur1 = self.fc1(spike_input[t])
            spk1, mem1 = self.lif1(cur1, mem1)
            spk1 = self.drop1(spk1)
            if self.hidden2:
                cur_mid = self.fc_mid(spk1)
                spk_mid, mem_mid = self.lif_mid(cur_mid, mem_mid)
                cur2 = self.fc2(spk_mid)
            else:
                cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            mem2_trace.append(mem2)
        # NOTE: classification reads out the *membrane potential* trace of
        # the output layer, not spike counts -- see the comment in the
        # original single-layer version's history for why spike-count
        # readout is fragile (it can collapse to a permanently-silent,
        # zero-gradient dead state). Membrane potential stays continuous
        # even while spikes are sparse.
        return torch.stack(mem2_trace, dim=0).mean(dim=0)  # (batch, n_classes)


def train_snn_presplit(X_spikes_train, y_train, X_spikes_test, y_test, n_classes,
                        hidden=256, hidden2=None, dropout=0.0,
                        epochs=120, lr=2e-3, weight_decay=1e-4, grad_clip=5.0,
                        seed=0, verbose=True):
    """
    Same model as train_snn, but takes an ALREADY-MADE train/test split
    instead of making its own. This matters once encoding depends on
    training-set statistics (adaptive thresholds) -- the split has to
    happen before encoding, not after, or the "test" set would leak
    into the numbers used to encode it.

    Adds three things the original fixed-30-epoch, no-schedule training
    loop didn't have, which matter more on noisier real CSI than on the
    clean synthetic data this was first tuned against:
      - a cosine LR schedule, so late training doesn't just bounce
        around a noisy full-batch gradient at a fixed step size
      - gradient clipping, standard practice for BPTT-style SNN
        training (this unrolls over T timesteps, same instability
        risk as an RNN)
      - weight decay, mild regularization against the higher-dimensional
        real feature space overfitting a training set this small

    Returns: (trained_model, final_train_acc, final_test_acc)
    """
    torch.manual_seed(seed)
    X_tr = np.stack(X_spikes_train)
    X_te = np.stack(X_spikes_test)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = DeltaSNN(input_dim=X_tr.shape[-1], hidden=hidden, hidden2=hidden2,
                      n_classes=n_classes, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()

    def to_tensor(X, y_):
        Xt = torch.tensor(X, dtype=torch.float32).permute(1, 0, 2).to(device)  # (T, N, 2D)
        yt = torch.tensor(np.asarray(y_), dtype=torch.long).to(device)
        return Xt, yt

    Xtr_t, ytr_t = to_tensor(X_tr, y_train)
    Xte_t, yte_t = to_tensor(X_te, y_test)

    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(Xtr_t)
        loss = loss_fn(out, ytr_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        opt.step()
        scheduler.step()
        if verbose and (ep % 10 == 0 or ep == epochs - 1):
            model.eval()
            with torch.no_grad():
                train_acc = (out.argmax(1) == ytr_t).float().mean().item()
                test_acc = (model(Xte_t).argmax(1) == yte_t).float().mean().item()
            print(f"[SNN] epoch {ep:03d}  loss={loss.item():.3f}  "
                  f"train_acc={train_acc:.3f}  test_acc={test_acc:.3f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}")

    model.eval()
    with torch.no_grad():
        final_train_acc = (model(Xtr_t).argmax(1) == ytr_t).float().mean().item()
        final_test_acc = (model(Xte_t).argmax(1) == yte_t).float().mean().item()
    return model, final_train_acc, final_test_acc


def train_snn(X_spikes, y, n_classes, hidden=256, epochs=30, lr=1e-3,
              seed=0, test_size=0.3, verbose=True):
    """
    Backward-compatible wrapper around train_snn_presplit: makes its own
    train/test split internally (fine when the encoding doesn't depend
    on training-set statistics, e.g. fixed-threshold or graded encoding).
    For adaptive-threshold encoding, split first and call
    train_snn_presplit directly instead -- see main.py.

    Returns: (trained_model, final_test_accuracy, (X_tr, X_te, y_tr, y_te))
    """
    X_arr = np.stack(X_spikes)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_arr, y, test_size=test_size, random_state=seed, stratify=y
    )
    model, _, test_acc = train_snn_presplit(
        X_tr, y_tr, X_te, y_te, n_classes=n_classes, hidden=hidden,
        epochs=epochs, lr=lr, seed=seed, verbose=verbose,
    )
    return model, test_acc, (X_tr, X_te, y_tr, y_te)


def snn_predict(model, X_spikes):
    """Run inference on a fresh list of (T, 2D) spike arrays (already
    padded to the same T the model was trained with)."""
    device = next(model.parameters()).device
    X_arr = np.stack(X_spikes)
    Xt = torch.tensor(X_arr, dtype=torch.float32).permute(1, 0, 2).to(device)
    model.eval()
    with torch.no_grad():
        preds = model(Xt).argmax(1).cpu().numpy()
    return preds

In [23]:
"""
main.py
-------
Orchestrates the full Part 2 pipeline:

  1. load data (synthetic by default -- flip USE_SYNTHETIC once you
     have a real Widar/SignFi download on disk)
  2. train the no-model k-NN fingerprint baseline
  3. train the SNN, with a choice of spike encoding strategy
  4. run the synthetic motion-corruption stress test on both

--------------------------------------------------------------------
Works in two setups:
  (a) as real separate .py files (e.g. your /code submission folder)
      -> the imports below succeed normally.
  (b) pasted into notebook cells, one file per cell, run in order:
      data_loader -> features -> baseline_knn -> snn_classifier
      -> stress_test -> main (this cell) -> a cell calling main()
      -> there's no data_loader.py etc. on disk for Python to find,
      so the imports below fail, and the except clause below just
      assumes those names already exist in the notebook's global
      namespace from the earlier cells having been run.

If you rerun a module cell after editing it, rerun this main cell too
before calling main() again -- case (b) doesn't re-import anything
for you, so it won't otherwise pick up the change.
--------------------------------------------------------------------

ENCODING_STRATEGY controls how packet-to-packet deltas become spikes:

  'fixed'    -- one global threshold for every channel (original version).
  'adaptive' -- per-channel threshold, calibrated from TRAINING data only
                (compute_adaptive_thresholds), then applied unchanged to
                both train and test. This is deliberately NOT a threshold
                that keeps adjusting itself at test time -- that would
                let it partially "adapt away" whatever you're testing
                against (e.g. corruption in the stress test), which
                would be measuring the wrong thing. What's adaptive here
                is the *calibration*, not an online moving target.
  'graded'   -- skip binarization; feed clipped delta magnitude directly
                as a graded input current instead of a spike/no-spike bit.

Verified on a synthetic scenario built specifically to isolate the
failure mode 'adaptive' targets (per-channel variance heterogeneity
that survives delta_sequence's global per-file normalization -- real
subcarriers do this, some sit near a fade and swing far more than
others): adaptive beat fixed 93.3% vs 83.3% test accuracy, same
architecture, same training budget. Not guaranteed to transfer 1:1 to
your real data, but it's evidence, not a guess -- and 'fixed' stays
one config line away if you want to A/B it yourself on your data.

Run (script version): python main.py
Run (notebook version): run this cell, then call main() in the next cell
"""

import numpy as np
from sklearn.model_selection import train_test_split

try:
    from data_loader import generate_synthetic_dataset, load_real_dataset, remap_labels
    from features import amplitude_snapshot, delta_sequence
    from baseline_knn import train_and_eval_knn
    from snn_classifier import (delta_modulate, compute_adaptive_thresholds,
                                 adaptive_delta_modulate, graded_delta_encode,
                                 pad_or_trim, train_snn_presplit, snn_predict)
    from stress_test import run_stress_test
except ImportError:
    # Setup (b): these files don't exist on disk as separate modules --
    # assume every name above is already defined in this notebook's
    # global namespace because its cell already ran. Nothing to do.
    pass

# ------------------------------------------------------------------
# CONFIG -- edit these first
# ------------------------------------------------------------------
USE_SYNTHETIC = False             # False once real data is downloaded
DATA_DIR = r"D:\RealDownloads\20181117\20181117\user4"      # only used if USE_SYNTHETIC = False
MAX_PACKETS_PER_FILE = 300       # truncate each real CSI file to this many packets
                                  # (memory lever #1 -- see data_loader.load_real_csi_file)
MAX_FILES = None                 # cap total files loaded, e.g. 1500 (memory lever #2);
                                  # None = load everything found
N_LOCATIONS = 5                  # only used for synthetic generation

ENCODING_STRATEGY = "adaptive"   # 'fixed' | 'adaptive' | 'graded' -- see docstring above
DELTA_THRESHOLD = 0.5            # used only when ENCODING_STRATEGY == 'fixed'
ADAPTIVE_K = 1.0                 # per-channel threshold = ADAPTIVE_K * that channel's train-set std
GRADED_CLIP = 3.0                # used only when ENCODING_STRATEGY == 'graded'
SPIKE_SEQ_LEN = 60               # pad/trim encoded sequences to this many steps

SNN_HIDDEN = 128
SNN_HIDDEN2 = 64                 # set to None for the original single-hidden-layer model
SNN_DROPOUT = 0.1
SNN_EPOCHS = 120
SNN_LR = 2e-3

TEST_SIZE = 0.3
SEED = 0


def encode_delta_sequence(delta_seq, thresholds=None):
    """Apply whichever ENCODING_STRATEGY is configured. `thresholds` is
    only used (and only meaningful) for 'adaptive' -- pass the array
    from compute_adaptive_thresholds, computed on the training split."""
    if ENCODING_STRATEGY == "fixed":
        return delta_modulate(delta_seq, threshold=DELTA_THRESHOLD)
    elif ENCODING_STRATEGY == "adaptive":
        return adaptive_delta_modulate(delta_seq, thresholds)
    elif ENCODING_STRATEGY == "graded":
        return graded_delta_encode(delta_seq, clip=GRADED_CLIP)
    else:
        raise ValueError(f"unknown ENCODING_STRATEGY: {ENCODING_STRATEGY}")


def main():
    # 1. Load data ----------------------------------------------------
    if USE_SYNTHETIC:
        print("[data] using synthetic CSI (edit USE_SYNTHETIC=False once real data is ready)")
        X_raw, y = generate_synthetic_dataset(n_locations=N_LOCATIONS, seed=SEED)
    else:
        X_raw, y = load_real_dataset(DATA_DIR, max_files=MAX_FILES, max_packets=MAX_PACKETS_PER_FILE)

    y = np.array(y)
    y, label_map = remap_labels(y)
    n_classes = len(label_map)
    print(f"[data] {len(X_raw)} samples, {n_classes} location classes")
    print(f"[data] label remap (original -> model index): {label_map}\n")

    # 2. No-model baseline ---------------------------------------------
    print("=== No-model baseline (k-NN fingerprint match) ===")
    X_amp = np.stack([amplitude_snapshot(a) for a in X_raw])
    knn_model, knn_acc, knn_cm, _ = train_and_eval_knn(
        X_amp, y, k=1, test_size=TEST_SIZE, seed=SEED, return_model=True
    )

    # 3. SNN on encoded CSI deltas ---------------------------------------
    # Split RAW sequences first, THEN compute deltas/thresholds/spikes.
    # This order matters: if ENCODING_STRATEGY == 'adaptive', the
    # per-channel thresholds are calibrated from compute_adaptive_thresholds,
    # and that must only ever see the training split -- fitting it on
    # data that includes the test set would be leakage, quietly inflating
    # the reported test accuracy.
    print(f"=== SNN (encoding strategy: '{ENCODING_STRATEGY}') ===")
    X_raw_tr, X_raw_te, y_tr, y_te = train_test_split(
        X_raw, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    delta_tr = [delta_sequence(a) for a in X_raw_tr]
    delta_te = [delta_sequence(a) for a in X_raw_te]

    thresholds = None
    if ENCODING_STRATEGY == "adaptive":
        thresholds = compute_adaptive_thresholds(delta_tr, k=ADAPTIVE_K)
        print(f"[encoding] adaptive thresholds: min={thresholds.min():.3f} "
              f"max={thresholds.max():.3f} (ratio {thresholds.max()/thresholds.min():.1f}x "
              f"-- a wide spread here means channels really do have very "
              f"different dynamic range, which is exactly what a single "
              f"global threshold can't handle)")

    X_spikes_tr = [pad_or_trim(encode_delta_sequence(d, thresholds), SPIKE_SEQ_LEN) for d in delta_tr]
    X_spikes_te = [pad_or_trim(encode_delta_sequence(d, thresholds), SPIKE_SEQ_LEN) for d in delta_te]

    snn_model, snn_train_acc, snn_acc = train_snn_presplit(
        X_spikes_tr, y_tr, X_spikes_te, y_te, n_classes=n_classes,
        hidden=SNN_HIDDEN, hidden2=SNN_HIDDEN2, dropout=SNN_DROPOUT,
        epochs=SNN_EPOCHS, lr=SNN_LR, seed=SEED,
    )
    print(f"[SNN] final train_acc={snn_train_acc:.3f}  test_acc={snn_acc:.3f}")
    if snn_train_acc - snn_acc > 0.25:
        print("[SNN] note: train accuracy is well above test accuracy -- that's "
              "overfitting, and the fix is more data/regularization/dropout, not "
              "a different threshold scheme.")
    elif snn_train_acc < 0.5:
        print("[SNN] note: train accuracy itself is low -- that's underfitting, "
              "the model isn't finding a fittable pattern even on data it's "
              "allowed to memorize. Worth checking whether the delta signal "
              "here is dominated by something uncorrelated with location "
              "(e.g. Widar's own gesture label) rather than blaming the "
              "threshold or optimizer.")

    # 4. Stress test: simulate a moving/vibrating collector --------------
    print("\n=== Stress test: static-trained classifiers vs. simulated motion ===")

    def eval_baseline(X_raw_list, y_list):
        feats = np.stack([amplitude_snapshot(a) for a in X_raw_list])
        preds = knn_model.predict(feats)
        return (preds == np.array(y_list)).mean()

    def eval_snn(X_raw_list, y_list):
        deltas = [delta_sequence(a) for a in X_raw_list]
        # NOTE: reuses the thresholds calibrated on the ORIGINAL training
        # split above, not refit on this (possibly corrupted) data -- this
        # represents a system that was calibrated once and then deployed,
        # which is the fair way to ask "how robust is it," not "how well
        # can it re-calibrate around the corruption."
        spikes = [pad_or_trim(encode_delta_sequence(d, thresholds), SPIKE_SEQ_LEN) for d in deltas]
        preds = snn_predict(snn_model, spikes)
        return (preds == np.array(y_list)).mean()

    run_stress_test(X_raw_te, y_te, eval_baseline, eval_snn)

    print("\n[done] baseline test acc = {:.3f}, SNN test acc = {:.3f}".format(knn_acc, snn_acc))


# --------------------------------------------------------------------
# Script version: `python main.py` runs this automatically (__name__
# is "__main__" for a script). Notebook version: __name__ is also
# "__main__" for a normally-run cell, so this fires too -- but if your
# notebook setup executes cells any other way (papermill, %run with
# args, etc.) and this silently doesn't fire, just call main() directly
# in the next cell instead.
# --------------------------------------------------------------------
if __name__ == "__main__":
    main()

[data] using synthetic CSI (edit USE_SYNTHETIC=False once real data is ready)
[data] 150 samples, 5 location classes
[data] label remap (original -> model index): {np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4}

=== No-model baseline (k-NN fingerprint match) ===
[k-NN baseline] k=1  test accuracy = 1.000
confusion matrix:
 [[9 0 0 0 0]
 [0 9 0 0 0]
 [0 0 9 0 0]
 [0 0 0 9 0]
 [0 0 0 0 9]]
=== SNN (encoding strategy: 'adaptive') ===
[encoding] adaptive thresholds: min=0.960 max=1.041 (ratio 1.1x -- a wide spread here means channels really do have very different dynamic range, which is exactly what a single global threshold can't handle)
[SNN] epoch 000  loss=1.728  train_acc=0.257  test_acc=0.244  lr=2.00e-03
[SNN] epoch 010  loss=1.584  train_acc=0.210  test_acc=0.333  lr=1.96e-03
[SNN] epoch 020  loss=1.507  train_acc=0.410  test_acc=0.267  lr=1.85e-03
[SNN] epoch 030  loss=1.422  train_acc=0.743  test_acc=0.622  lr=1.69e-03
[SNN] epoch 040  loss=1.237 